In [9]:
import boto3
import json

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

text = "David lives in Fuquay-Varina, North Carolina. He is working on AI reliability labs with Bedrock."

body = json.dumps({"inputText": text})

response = bedrock.invoke_model(
    modelId="amazon.titan-embed-text-v2:0",
    contentType="application/json",
    accept="application/json",
    body=body
)

embedding = json.loads(response['body'].read())['embedding']
print(f"Embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Embedding dimension: 1024
First 5 values: [-0.10151564329862595, 0.10864506661891937, -0.008325809612870216, -0.02100762352347374, -0.03148782253265381]


# Day 22: Titan Embeddings v2 Test

Quick test of `amazon.titan-embed-text-v2:0` using local Jupyter + boto3.

- Model: 1024 dimensions
- Goal: Verify Bedrock access and get a sample embedding for future RAG work

In [10]:
texts = [
    "David lives in Fuquay-Varina, North Carolina.",
    "He is building AI reliability labs with AWS Bedrock.",
    "Fuquay-Varina is in Wake County."
]

embeddings = []
for t in texts:
    body = json.dumps({"inputText": t})
    resp = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        contentType="application/json",
        accept="application/json",
        body=body
    )
    emb = json.loads(resp['body'].read())['embedding']
    embeddings.append(emb)
    print(f"Text: {t[:30]}... → {len(emb)} dims")

print(f"Got {len(embeddings)} embeddings")

Text: David lives in Fuquay-Varina, ... → 1024 dims
Text: He is building AI reliability ... → 1024 dims
Text: Fuquay-Varina is in Wake Count... → 1024 dims
Got 3 embeddings


In [11]:
from numpy import dot
from numpy.linalg import norm

def cosine_sim(a, b):
    return dot(a, b) / (norm(a) * norm(b))

# Compare first and third
sim = cosine_sim(embeddings[0], embeddings[2])
print(f"Cosine similarity between text 1 and text 3: {sim:.4f}")

Cosine similarity between text 1 and text 3: 0.5917


In [12]:
# Quick check + more comparisons
if 'embeddings' in globals() and len(embeddings) >= 3:
    print("Embeddings list is ready (length:", len(embeddings), ")")
    
    # Compare 1 vs 2
    sim12 = cosine_sim(embeddings[0], embeddings[1])
    print(f"Text 1 vs Text 2: {sim12:.4f}")
    
    # Compare 1 vs 3
    sim13 = cosine_sim(embeddings[0], embeddings[2])
    print(f"Text 1 vs Text 3: {sim13:.4f}")
    
    # Compare 2 vs 3
    sim23 = cosine_sim(embeddings[1], embeddings[2])
    print(f"Text 2 vs Text 3: {sim23:.4f}")
else:
    print("Run the embedding generation cell first!")

Embeddings list is ready (length: 3 )
Text 1 vs Text 2: 0.1049
Text 1 vs Text 3: 0.5917
Text 2 vs Text 3: 0.0296
